# Step 4: Temporal Modeling — Full Pipeline

## Sub-pipelines
| Sub-step | Method | Inputs | Output Column |
|---|---|---|---|
| 4A | **TransE** (static KG embedding) | `knowledge_edges.csv`, `entity_nodes.csv`, `paper_nodes.csv` | `transe_unexpectedness` |
| 4B | **TGN-approx** (temporal link prediction via abstract embeddings) | `abstract_embeddings.npy`, `paper_ids.npy`, `citation_edges.csv`, `citation_edges_2022_2025.csv` | `tgn_unexpectedness` |
| Combine | Disruptiveness score | All above + `novelty_feature_matrix_with_score.csv` | `disruptiveness` |

**Disruptiveness formula:**
$$D(p) = 0.3 \times \underbrace{\frac{1}{2}(U_{TransE}(p) + U_{TGN}(p))}_{\text{KG unexpectedness}} + 0.7 \times N_s(p)$$

where $N_s(p)$ = `composite_novelty` from Step 3.

## Cell 0 — Imports & Config

In [27]:
import os
import time
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')
np.random.seed(42)
BASE_PATH = Path('../../outputs')
# ── Input paths ───────────────────────────────────────────────────────────────
KNOWLEDGE_EDGES_PATH   = BASE_PATH/'final'/'knowledge_edges.csv'
ENTITY_NODES_PATH      = BASE_PATH/'final'/'entity_nodes.csv'
PAPER_NODES_PATH       = BASE_PATH/'final'/'paper_nodes.csv'
NOVELTY_MATRIX_PATH    = BASE_PATH/'final'/'novelty_feature_matrix_with_score.csv'
ABSTRACT_EMB_PATH      = BASE_PATH/'final'/'abstract_embeddings.npy'
PAPER_IDS_PATH         = BASE_PATH/'final'/'paper_ids.npy'
CITATION_EDGES_PATH    = BASE_PATH/'other'/'citation_edges.csv'
CITATION_EDGES_22_PATH = BASE_PATH/'other'/'citation_edges_2022_2025.csv'

# ── TransE hyperparameters ────────────────────────────────────────────────────
TRANSE_DIM    = 128    # embedding dimension
TRANSE_MARGIN = 1.0    # hinge loss margin γ
TRANSE_LR     = 0.01   # SGD learning rate
TRANSE_EPOCHS = 200    # training epochs
TRANSE_BATCH  = 1024   # batch size

# ── TGN hyperparameters ───────────────────────────────────────────────────────
TGN_DECAY_LAMBDA = 0.3  # exponential decay rate for temporal memory
TGN_REF_YEAR     = 2021 # reference year (novel paper era boundary)

# ── Disruptiveness weights ────────────────────────────────────────────────────
LAMBDA_KG_UNEXPECTEDNESS = 0.30
LAMBDA_SEMANTIC_NOVELTY  = 0.70

# ── Output path ───────────────────────────────────────────────────────────────
OUTPUT_PATH = BASE_PATH/'final'/'novelty_feature_matrix_with_score.csv'
TRANSE_SCORES_PATH = BASE_PATH/'final'/'transe_unexpected_scores.csv'
TGN_SCORES_PATH    = BASE_PATH/'final'/'tgn_unexpected_scores.csv'
COMBINED_PATH      = BASE_PATH/'final'/'combined_disruptiveness.csv'

def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}')

def minmax_norm(arr):
    mn, mx = np.nanmin(arr), np.nanmax(arr)
    if mx - mn < 1e-9:
        return np.zeros_like(arr, dtype=float)
    return (arr - mn) / (mx - mn)

def l2_norm_rows(mat):
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    return mat / np.where(norms < 1e-9, 1.0, norms)

log('Config loaded.')

[14:55:21] Config loaded.


## Cell 1 — Load All Input Files

In [28]:
log('Loading input files...')

# ── Knowledge graph ───────────────────────────────────────────────────────────
ke = pd.read_csv(KNOWLEDGE_EDGES_PATH)
ke['year'] = ke['year'].fillna(ke['year'].median())
log(f'  knowledge_edges     : {len(ke):,} edges | cols: {ke.columns.tolist()}')

# ── Nodes ─────────────────────────────────────────────────────────────────────
entity_nodes = pd.read_csv(ENTITY_NODES_PATH)
paper_nodes  = pd.read_csv(PAPER_NODES_PATH)
log(f'  entity_nodes        : {len(entity_nodes):,} entities')
log(f'  paper_nodes         : {len(paper_nodes):,} papers | splits: {paper_nodes["split"].value_counts().to_dict()}')

# ── Step 3 novelty matrix ─────────────────────────────────────────────────────
novelty_matrix = pd.read_csv(NOVELTY_MATRIX_PATH)
log(f'  novelty_matrix      : {len(novelty_matrix):,} papers | cols: {novelty_matrix.columns.tolist()}')

# ── Abstract embeddings ───────────────────────────────────────────────────────
abstract_emb = np.load(ABSTRACT_EMB_PATH, allow_pickle=True)
paper_ids    = np.load(PAPER_IDS_PATH, allow_pickle=True)
log(f'  abstract_embeddings : shape {abstract_emb.shape} | aligned with {len(paper_ids)} paper_ids')

# ── Citation edges ────────────────────────────────────────────────────────────
cite_hist   = pd.read_csv(CITATION_EDGES_PATH)
cite_recent = pd.read_csv(CITATION_EDGES_22_PATH)
log(f'  citation_edges      : {len(cite_hist):,} (historical)')
log(f'  citation_edges_22+  : {len(cite_recent):,} (2022-2025)')

# ── Build embedding lookup dict ───────────────────────────────────────────────
abstract_emb_norm = l2_norm_rows(abstract_emb.astype(np.float32))
emb_lookup = {pid: abstract_emb_norm[i] for i, pid in enumerate(paper_ids)}
log(f'  Embedding lookup built: {len(emb_lookup):,} entries')

# ── Identify split sets ───────────────────────────────────────────────────────
skg_papers   = set(paper_nodes[paper_nodes['split'] == 'SKG']['node_id'])
novel_papers = set(paper_nodes[paper_nodes['split'] == 'NOVEL']['node_id'])
blog_papers  = set(paper_nodes[paper_nodes['split'] == 'BLOG']['node_id'])
log(f'  SKG: {len(skg_papers)} | NOVEL: {len(novel_papers)} | BLOG: {len(blog_papers)}')

[14:55:21] Loading input files...
[14:55:21]   knowledge_edges     : 577,024 edges | cols: ['source', 'target', 'predicate', 'year']
[14:55:21]   entity_nodes        : 293,149 entities
[14:55:21]   paper_nodes         : 4,860 papers | splits: {'SKG': 3166, 'NOVEL': 854, 'BLOG': 840}
[14:55:21]   novelty_matrix      : 4,242 papers | cols: ['paper_id', 'year', 'semantic_knn', 'structural_novelty', 'triple_count', 'cited_by_count', 'reference_count', 'citation_signal', 'struct_norm', 'semantic_norm', 'citation_norm', 'composite_novelty']
[14:55:21]   abstract_embeddings : shape (4242, 768) | aligned with 4242 paper_ids
[14:55:22]   citation_edges      : 7,425 (historical)
[14:55:22]   citation_edges_22+  : 320,724 (2022-2025)
[14:55:22]   Embedding lookup built: 4,242 entries
[14:55:22]   SKG: 3166 | NOVEL: 854 | BLOG: 840


## Cell 2 — TransE: Vocabulary & Triple Encoding

**Training split**: All knowledge edges where `source ∈ SKG_papers` → historical graph.

**Test split**: Edges where `source ∈ NOVEL_papers` → scored for unexpectedness.

**Node vocab**: union of `entity_nodes.node_id` ∪ `paper_nodes.node_id`.

**TransE score**: $f(h, r, t) = \|\mathbf{h} + \mathbf{r} - \mathbf{t}\|_2$ — lower = more plausible.

In [29]:
log('Building TransE vocabulary...')

# ── Entity vocab: entity_nodes + paper_nodes ───────────────────────────────────
all_node_ids = list(entity_nodes['node_id']) + list(paper_nodes['node_id'])
ent2id = {n: i for i, n in enumerate(dict.fromkeys(all_node_ids))}  # deduped
UNSEEN_ENT = len(ent2id)   # sentinel for unseen entities at test time

# ── Relation vocab: all predicates in knowledge_edges ─────────────────────────
all_preds = ke['predicate'].unique()
rel2id = {r: i for i, r in enumerate(all_preds)}
UNSEEN_REL = len(rel2id)   # sentinel for unseen predicates at test time

n_ent = len(ent2id) + 1   # +1 for UNSEEN sentinel
n_rel = len(rel2id) + 1   # +1 for UNSEEN sentinel

log(f'  Entities : {len(ent2id):,}  (+ 1 UNSEEN sentinel = {n_ent:,})')
log(f'  Relations: {len(rel2id):,}  (+ 1 UNSEEN sentinel = {n_rel:,})')

# ── Split knowledge_edges into train (SKG) and test (NOVEL) ───────────────────
ke_train = ke[ke['source'].isin(skg_papers)].reset_index(drop=True)
ke_test  = ke[ke['source'].isin(novel_papers)].reset_index(drop=True)
log(f'  KE train (SKG)  : {len(ke_train):,} edges')
log(f'  KE test  (NOVEL): {len(ke_test):,} edges  |  {ke_test["source"].nunique()} papers')

# ── Encode triples to integer IDs ─────────────────────────────────────────────
def encode(df, ent2id, rel2id, unseen_ent, unseen_rel):
    h = df['source'].map(lambda x: ent2id.get(x, unseen_ent)).values
    r = df['predicate'].map(lambda x: rel2id.get(x, unseen_rel)).values
    t = df['target'].map(lambda x: ent2id.get(x, unseen_ent)).values
    return np.column_stack([h, r, t]).astype(np.int32)

train_triples = encode(ke_train, ent2id, rel2id, UNSEEN_ENT, UNSEEN_REL)
test_triples  = encode(ke_test,  ent2id, rel2id, UNSEEN_ENT, UNSEEN_REL)

# Filter train: remove any triple containing the UNSEEN sentinel
valid_train_mask = np.all(train_triples < np.array([n_ent-1, n_rel-1, n_ent-1]), axis=1)
train_triples = train_triples[valid_train_mask]
log(f'  Valid train triples: {len(train_triples):,} (after filtering UNSEEN)')
log(f'  Test triples       : {len(test_triples):,}')

[14:55:22] Building TransE vocabulary...
[14:55:22]   Entities : 298,009  (+ 1 UNSEEN sentinel = 298,010)
[14:55:22]   Relations: 18,494  (+ 1 UNSEEN sentinel = 18,495)
[14:55:22]   KE train (SKG)  : 378,527 edges
[14:55:22]   KE test  (NOVEL): 82,599 edges  |  528 papers
[14:55:22]   Valid train triples: 378,527 (after filtering UNSEEN)
[14:55:22]   Test triples       : 82,599


## Cell 3 — TransE: Training Loop (NumPy)

**Objective** (margin-based ranking loss):
$$\mathcal{L} = \sum_{(h,r,t) \in \mathcal{S}} \sum_{(h',r,t') \in \mathcal{S}'} \max\left(0,\; \gamma + f(h,r,t) - f(h',r,t')\right)$$

Entity embeddings are constrained to unit $L_2$ sphere after each gradient step.

In [30]:
log(f'Training TransE: dim={TRANSE_DIM}, margin={TRANSE_MARGIN}, lr={TRANSE_LR}, epochs={TRANSE_EPOCHS}')

# ── Xavier initialization ─────────────────────────────────────────────────────
scale = np.sqrt(6.0 / TRANSE_DIM)
ent_emb = np.random.uniform(-scale, scale, (n_ent, TRANSE_DIM)).astype(np.float32)
rel_emb = np.random.uniform(-scale, scale, (n_rel, TRANSE_DIM)).astype(np.float32)
ent_emb = l2_norm_rows(ent_emb)

N = len(train_triples)
loss_history = []

for epoch in range(TRANSE_EPOCHS):
    idx = np.random.permutation(N)
    triples = train_triples[idx]
    epoch_loss = 0.0
    n_batches  = 0

    for start in range(0, N, TRANSE_BATCH):
        batch = triples[start:start + TRANSE_BATCH]
        B = len(batch)
        h_ids, r_ids, t_ids = batch[:, 0], batch[:, 1], batch[:, 2]

        # Negative sampling: corrupt head (50%) or tail (50%)
        corrupt_head = np.random.rand(B) < 0.5
        neg_h = h_ids.copy()
        neg_t = t_ids.copy()
        neg_h[ corrupt_head] = np.random.randint(0, n_ent - 1, corrupt_head.sum())
        neg_t[~corrupt_head] = np.random.randint(0, n_ent - 1, (~corrupt_head).sum())

        # Fetch embeddings
        h_v  = ent_emb[h_ids];  r_v = rel_emb[r_ids];  t_v = ent_emb[t_ids]
        nh_v = ent_emb[neg_h]; nt_v = ent_emb[neg_t]

        # Scores
        pos_d = h_v + r_v - t_v
        neg_d = nh_v + r_v - nt_v
        pos_s = np.linalg.norm(pos_d, axis=1)
        neg_s = np.linalg.norm(neg_d, axis=1)

        # Hinge loss
        loss_v = np.maximum(0.0, TRANSE_MARGIN + pos_s - neg_s)
        active = loss_v > 0
        epoch_loss += loss_v.sum()
        n_batches  += 1

        if not active.any():
            continue

        # Subgradients of ||x||₂:  ∂/∂x = x / ||x||₂
        def g(d, s):
            s_ = np.where(s < 1e-9, 1.0, s)[:, None]
            return d / s_

        pg = g(pos_d, pos_s)
        ng = g(neg_d, neg_s)
        a  = active

        # Gradient updates (positive triple)
        np.add.at(ent_emb, h_ids[a], -TRANSE_LR * pg[a])
        np.add.at(rel_emb, r_ids[a], -TRANSE_LR * pg[a])
        np.add.at(ent_emb, t_ids[a],  TRANSE_LR * pg[a])
        # Gradient updates (negative triple)
        np.add.at(ent_emb, neg_h[a],  TRANSE_LR * ng[a])
        np.add.at(rel_emb, r_ids[a],  TRANSE_LR * ng[a])
        np.add.at(ent_emb, neg_t[a], -TRANSE_LR * ng[a])

        # Re-normalize updated entity embeddings to unit sphere
        touched = np.unique(np.concatenate([h_ids[a], t_ids[a], neg_h[a], neg_t[a]]))
        ent_emb[touched] = l2_norm_rows(ent_emb[touched])

    avg = epoch_loss / max(n_batches, 1)
    loss_history.append(avg)
    if (epoch + 1) % 40 == 0 or epoch == 0:
        log(f'  Epoch {epoch+1:3d}/{TRANSE_EPOCHS} | loss = {avg:.4f}')

log('TransE training complete.')

[14:55:22] Training TransE: dim=128, margin=1.0, lr=0.01, epochs=200
[14:55:28]   Epoch   1/200 | loss = 991.8690
[14:57:33]   Epoch  40/200 | loss = 302.0377
[14:59:09]   Epoch  80/200 | loss = 199.0434
[15:00:30]   Epoch 120/200 | loss = 148.9761
[15:01:41]   Epoch 160/200 | loss = 120.1855
[15:02:46]   Epoch 200/200 | loss = 103.4281
[15:02:46] TransE training complete.


## Cell 4 — TransE: Score Novel Papers

For each NOVEL paper $p$, aggregate triple scores:
$$U_{TransE}(p) = \frac{1}{|T_p|} \sum_{(h,r,t) \in T_p} \|\mathbf{h} + \mathbf{r} - \mathbf{t}\|_2$$

Unseen entity/relation → maximum distance score (= fully unexpected).

Final score normalized to $[0, 1]$.

In [31]:
log('Scoring novel papers with TransE...')

MAX_DIST = 10.0   # distance assigned to triples with UNSEEN entities/relations

def transe_score(h_ids, r_ids, t_ids):
    """
    Vectorised TransE scoring.
    Triples with unseen sentinel IDs get MAX_DIST.
    """
    scores = np.full(len(h_ids), MAX_DIST, dtype=np.float32)
    mask = (h_ids < n_ent - 1) & (r_ids < n_rel - 1) & (t_ids < n_ent - 1)
    if mask.any():
        hv = ent_emb[h_ids[mask]]
        rv = rel_emb[r_ids[mask]]
        tv = ent_emb[t_ids[mask]]
        scores[mask] = np.linalg.norm(hv + rv - tv, axis=1)
    return scores

# Attach paper_id back to test triples
ke_test_scored = ke_test.copy()
h_arr = test_triples[:, 0]
r_arr = test_triples[:, 1]
t_arr = test_triples[:, 2]
ke_test_scored['transe_dist']    = transe_score(h_arr, r_arr, t_arr)
ke_test_scored['source_unseen']  = (h_arr == UNSEEN_ENT).astype(int)
ke_test_scored['target_unseen']  = (t_arr == UNSEEN_ENT).astype(int)
ke_test_scored['pred_unseen']    = (r_arr == UNSEEN_REL).astype(int)
ke_test_scored['any_unseen']     = (
    ke_test_scored['source_unseen'] | ke_test_scored['target_unseen'] | ke_test_scored['pred_unseen']
).astype(int)

# Aggregate per NOVEL paper
transe_paper = ke_test_scored.groupby('source').agg(
    n_triples         = ('transe_dist', 'count'),
    mean_dist         = ('transe_dist', 'mean'),
    min_dist          = ('transe_dist', 'min'),
    max_dist          = ('transe_dist', 'max'),
    std_dist          = ('transe_dist', 'std'),
    pct_unseen_entity = ('any_unseen',  'mean'),
    pct_unseen_pred   = ('pred_unseen', 'mean'),
).reset_index().rename(columns={'source': 'paper_id'})

transe_paper['transe_unexpectedness'] = minmax_norm(transe_paper['mean_dist'].values)

log(f'  Scored {len(transe_paper)} NOVEL papers')
log(f'  Mean TransE distance  : {transe_paper["mean_dist"].mean():.4f}')
log(f'  Mean unexpectedness   : {transe_paper["transe_unexpectedness"].mean():.4f}')
log(f'  Papers >50% unseen ent: {(transe_paper["pct_unseen_entity"] >= 0.5).sum()}')

transe_paper.to_csv(TRANSE_SCORES_PATH, index=False)
log(f'  Saved → {TRANSE_SCORES_PATH}')
transe_paper.head()

[15:02:46] Scoring novel papers with TransE...
[15:02:46]   Scored 528 NOVEL papers
[15:02:46]   Mean TransE distance  : 2.0366
[15:02:46]   Mean unexpectedness   : 0.5217
[15:02:46]   Papers >50% unseen ent: 0
[15:02:46]   Saved → ..\..\outputs\final\transe_unexpected_scores.csv


,paper_id,n_triples,mean_dist,min_dist,max_dist,std_dist,pct_unseen_entity,pct_unseen_pred,transe_unexpectedness
0,NOVEL_DIA_0,99,2.223738,1.480948,2.737427,0.475629,0.0,0.0,0.710063
1,NOVEL_DIA_1,132,2.068059,1.400753,2.769335,0.484481,0.0,0.0,0.553354
2,NOVEL_DIA_10,134,2.090899,1.456547,2.808419,0.452151,0.0,0.0,0.576344
3,NOVEL_DIA_11,67,2.083574,1.434995,2.734103,0.447658,0.0,0.0,0.568971
4,NOVEL_DIA_12,132,2.051442,1.444224,2.709960,0.470176,0.0,0.0,0.536626


## Cell 5 — TGN Approximation: Temporal Memory Construction

**Design**: Since no GPU/PyTorch is available, we implement a discrete TGN approximation:

For each node $v$ at reference time $T$:
$$\text{memory}(v) = \sum_{t \leq T} e^{-\lambda(T-t)} \cdot \mathbf{1}[v \text{ appears at time } t]$$

For **link prediction**, we use abstract SBERT embeddings as node state $z_v$:
$$P(u \to v \mid \text{history}) \propto \text{cosim}(z_u, z_v) \times w_{temporal}(u, v)$$

where $w_{temporal}$ = exponentially decayed co-citation frequency between $u$ and $v$ in the historical citation graph.

**Unexpectedness** of a citation $p \to q$:
$$U_{TGN}(p, q) = 1 - P(p \to q \mid \text{history})$$

Aggregated per paper: mean over all outgoing citations.

In [32]:
log('Building TGN temporal memory from historical citation graph...')

# ── Step 1: Build temporal co-citation frequency matrix (node-pair level) ─────
# For all historical edges (source → target, year), compute decayed frequency
# per (source, target) pair.

# Use combined citation edges: cite_hist (pre-2022) + cite_recent (2022+)
# Train memory on cite_hist only (historical), score NOVEL papers from cite_recent
cite_hist_clean = cite_hist.copy()
cite_hist_clean['year'] = pd.to_numeric(cite_hist_clean['year'], errors='coerce').fillna(2018)

# Temporal decay weight for each edge in historical graph
cite_hist_clean['decay_w'] = np.exp(
    -TGN_DECAY_LAMBDA * (TGN_REF_YEAR - cite_hist_clean['year'].clip(upper=TGN_REF_YEAR))
)

# Node activity: total decayed in-degree per target node
# (how much a paper is cited historically → proxy for expected future citations)
node_activity = cite_hist_clean.groupby('target')['decay_w'].sum().to_dict()
total_activity = sum(node_activity.values()) + 1e-9

# Pair-level decayed frequency: P(source → target | history)
pair_freq = cite_hist_clean.groupby(['source', 'target'])['decay_w'].sum()
pair_freq_dict = pair_freq.to_dict()  # {(src, tgt): decayed_freq}

# Source-level total outgoing frequency (normalization)
src_total = cite_hist_clean.groupby('source')['decay_w'].sum().to_dict()

log(f'  Historical citation pairs : {len(pair_freq_dict):,}')
log(f'  Unique targets (cited)    : {len(node_activity):,}')
log(f'  Year range                : {cite_hist_clean["year"].min():.0f} – {cite_hist_clean["year"].max():.0f}')

[15:02:46] Building TGN temporal memory from historical citation graph...
[15:02:46]   Historical citation pairs : 7,425
[15:02:46]   Unique targets (cited)    : 1,257
[15:02:46]   Year range                : 2010 – 2022


## Cell 6 — TGN: Score NOVEL Papers

For each NOVEL paper $p$ and each citation $p \to q$:

$$\text{link\_prob}(p \to q) = \alpha \cdot \text{cosim}(z_p, z_q) + (1-\alpha) \cdot \frac{f(p, q)}{Z_p}$$

where:
- $\text{cosim}(z_p, z_q)$ = cosine similarity of SBERT abstracts (embedding space signal)
- $f(p, q)$ = exponentially decayed co-citation frequency (temporal memory signal)
- $Z_p$ = total outgoing citation weight of $p$ in history
- $\alpha = 0.6$ (embedding-dominant)

For targets unseen in historical graph: $f(p,q) = 0$ (pure embedding signal).

In [33]:
log('Scoring NOVEL papers with TGN-approximation...')

ALPHA = 0.6   # weight on embedding similarity vs temporal frequency

# NOVEL papers to score: take from both citation edge files
novel_cite_test = pd.concat([
    cite_hist[cite_hist['source'].isin(novel_papers)],
    cite_recent[cite_recent['source'].isin(novel_papers)]
], ignore_index=True).drop_duplicates()

log(f'  NOVEL citation edges to score: {len(novel_cite_test):,} from {novel_cite_test["source"].nunique()} papers')

tgn_scores_rows = []

for paper_id, group in novel_cite_test.groupby('source'):
    targets = group['target'].values
    p_emb = emb_lookup.get(paper_id, None)
    link_probs = []

    for tgt in targets:
        # ── Cosine similarity component ──────────────────────────────────────
        q_emb = emb_lookup.get(tgt, None)
        if p_emb is not None and q_emb is not None:
            cosim = float(np.dot(p_emb, q_emb))   # both unit-normalized
            cosim = (cosim + 1.0) / 2.0            # shift to [0, 1]
        else:
            cosim = 0.5   # neutral if embedding missing

        # ── Temporal frequency component ─────────────────────────────────────
        hist_freq = pair_freq_dict.get((paper_id, tgt), 0.0)
        src_norm  = src_total.get(paper_id, 1.0)
        freq_prob = hist_freq / max(src_norm, 1e-9)
        freq_prob = min(freq_prob, 1.0)

        # ── Combined link probability ─────────────────────────────────────────
        lp = ALPHA * cosim + (1.0 - ALPHA) * freq_prob
        link_probs.append(lp)

    mean_lp = float(np.mean(link_probs)) if link_probs else 0.5
    tgn_scores_rows.append({
        'paper_id'          : paper_id,
        'n_citations'       : len(targets),
        'mean_link_prob'    : mean_lp,
        'min_link_prob'     : float(np.min(link_probs)) if link_probs else 0.5,
        'max_link_prob'     : float(np.max(link_probs)) if link_probs else 0.5,
        'tgn_raw_unexp'     : 1.0 - mean_lp,
    })

tgn_paper = pd.DataFrame(tgn_scores_rows)
tgn_paper['tgn_unexpectedness'] = minmax_norm(tgn_paper['tgn_raw_unexp'].values)

log(f'  Scored {len(tgn_paper)} NOVEL papers')
log(f'  Mean link probability   : {tgn_paper["mean_link_prob"].mean():.4f}')
log(f'  Mean TGN unexpectedness : {tgn_paper["tgn_unexpectedness"].mean():.4f}')

tgn_paper.to_csv(TGN_SCORES_PATH, index=False)
log(f'  Saved → {TGN_SCORES_PATH}')
tgn_paper.head()

[15:02:46] Scoring NOVEL papers with TGN-approximation...
[15:02:46]   NOVEL citation edges to score: 50,370 from 789 papers
[15:02:46]   Scored 789 NOVEL papers
[15:02:46]   Mean link probability   : 0.4820
[15:02:46]   Mean TGN unexpectedness : 0.7330
[15:02:46]   Saved → ..\..\outputs\final\tgn_unexpected_scores.csv


,paper_id,n_citations,mean_link_prob,min_link_prob,max_link_prob,tgn_raw_unexp,tgn_unexpectedness
0,NOVEL_DIA_1,16,0.590914,0.554154,0.609616,0.409086,0.573150
1,NOVEL_DIA_10,6,0.637911,0.593437,0.650990,0.362089,0.504193
2,NOVEL_DIA_12,1,0.974263,0.974263,0.974263,0.025737,0.010672
3,NOVEL_DIA_13,2,0.778182,0.776964,0.779401,0.221818,0.298376
4,NOVEL_DIA_14,4,0.653608,0.627939,0.680651,0.346392,0.481161


## Cell 7 — Merge into `novelty_feature_matrix_with_score.csv`

New columns added to the feature matrix:

| Column | Description |
|---|---|
| `transe_unexpectedness` | Normalized TransE distance — structural KG surprise |
| `tgn_unexpectedness` | Normalized TGN link improbability — temporal citation surprise |
| `kg_unexpectedness` | $0.5 \times$ TransE $+ 0.5 \times$ TGN |
| `disruptiveness` | $0.3 \times$ KG unexpectedness $+ 0.7 \times$ composite_novelty |
| `disruption_class` | DISRUPTIVE / TRANSITIONAL / INCREMENTAL |

In [34]:
log('Merging temporal scores into novelty_feature_matrix_with_score.csv...')

out = novelty_matrix.copy()

# ── Merge TransE scores ───────────────────────────────────────────────────────
transe_merge = transe_paper[['paper_id', 'n_triples', 'mean_dist', 'min_dist',
                              'pct_unseen_entity', 'pct_unseen_pred',
                              'transe_unexpectedness']]
out = out.merge(transe_merge, on='paper_id', how='left')
n_transe_filled = out['transe_unexpectedness'].notna().sum()
log(f'  TransE scores matched: {n_transe_filled} / {len(out)} papers')

# ── Merge TGN scores ──────────────────────────────────────────────────────────
tgn_merge = tgn_paper[['paper_id', 'n_citations', 'mean_link_prob',
                        'tgn_unexpectedness']]
out = out.merge(tgn_merge, on='paper_id', how='left')
n_tgn_filled = out['tgn_unexpectedness'].notna().sum()
log(f'  TGN scores matched  : {n_tgn_filled} / {len(out)} papers')

# ── Fill missing scores: SKG papers without NOVEL-specific scores ─────────────
# For papers outside NOVEL split: set unexpectedness = 0.0 (fully expected baseline)
out['transe_unexpectedness'] = out['transe_unexpectedness'].fillna(0.0)
out['tgn_unexpectedness']    = out['tgn_unexpectedness'].fillna(0.0)

# ── Combined KG unexpectedness ────────────────────────────────────────────────
out['kg_unexpectedness'] = 0.5 * out['transe_unexpectedness'] + 0.5 * out['tgn_unexpectedness']

# ── Disruptiveness score ──────────────────────────────────────────────────────
# D(p) = 0.3 × kg_unexpectedness + 0.7 × composite_novelty (from Step 3)
out['disruptiveness'] = (
    LAMBDA_KG_UNEXPECTEDNESS * out['kg_unexpectedness'] +
    LAMBDA_SEMANTIC_NOVELTY  * out['composite_novelty']
)
out['disruptiveness'] = minmax_norm(out['disruptiveness'].values)

# ── Classify ──────────────────────────────────────────────────────────────────
def classify(score):
    if score >= 0.6:  return 'DISRUPTIVE'
    if score >= 0.4:  return 'TRANSITIONAL'
    return 'INCREMENTAL'

out['disruption_class'] = out['disruptiveness'].apply(classify)

log(f'\n  Disruption class distribution:')
log(f'    DISRUPTIVE   : {(out["disruption_class"]=="DISRUPTIVE").sum()}')
log(f'    TRANSITIONAL : {(out["disruption_class"]=="TRANSITIONAL").sum()}')
log(f'    INCREMENTAL  : {(out["disruption_class"]=="INCREMENTAL").sum()}')
log(f'\n  Disruptiveness stats:')
log(f'    mean={out["disruptiveness"].mean():.4f}  std={out["disruptiveness"].std():.4f}')
log(f'    max={out["disruptiveness"].max():.4f}   min={out["disruptiveness"].min():.4f}')

[15:02:46] Merging temporal scores into novelty_feature_matrix_with_score.csv...
[15:02:46]   TransE scores matched: 496 / 4242 papers
[15:02:46]   TGN scores matched  : 703 / 4242 papers
[15:02:46] 
  Disruption class distribution:
[15:02:46]     DISRUPTIVE   : 3071
[15:02:46]     TRANSITIONAL : 1086
[15:02:46]     INCREMENTAL  : 85
[15:02:46] 
  Disruptiveness stats:
[15:02:46]     mean=0.6312  std=0.1157
[15:02:46]     max=1.0000   min=0.0000


In [35]:
# Recompute class boundaries using percentile-based thresholds
p75 = out['disruptiveness'].quantile(0.75)
p50 = out['disruptiveness'].quantile(0.50)

def classify_adaptive(score):
    if score >= p75: return 'DISRUPTIVE'
    if score >= p50: return 'TRANSITIONAL'
    return 'INCREMENTAL'

out['disruption_class'] = out['disruptiveness'].apply(classify_adaptive)

## Cell 8 — Save All Outputs

In [36]:
log('Saving outputs...')

# ── Primary output: merged feature matrix ─────────────────────────────────────
out.to_csv(OUTPUT_PATH, index=False)
log(f'  ✓ novelty_feature_matrix_with_score.csv  ({len(out):,} rows, {len(out.columns)} cols)')

# ── Standalone score files ─────────────────────────────────────────────────────
# (already saved in Cells 4 & 6)
log(f'  ✓ {TRANSE_SCORES_PATH}  ({len(transe_paper)} papers)')
log(f'  ✓ {TGN_SCORES_PATH}    ({len(tgn_paper)} papers)')

# ── Combined disruptiveness: NOVEL papers only ─────────────────────────────────
novel_out = out[out['paper_id'].str.startswith('NOVEL_')].copy()
novel_out = novel_out.sort_values('disruptiveness', ascending=False).reset_index(drop=True)
novel_out['rank'] = range(1, len(novel_out) + 1)
novel_out.to_csv(COMBINED_PATH, index=False)
log(f'  ✓ {COMBINED_PATH}  ({len(novel_out)} NOVEL papers)')

# ── TransE training loss ───────────────────────────────────────────────────────
pd.DataFrame({'epoch': range(1, len(loss_history)+1), 'loss': loss_history}).to_csv(
    'transe_training_loss.csv', index=False)
log(f'  ✓ transe_training_loss.csv')

print()
print('=== TOP 10 MOST DISRUPTIVE NOVEL PAPERS ===')
cols = ['rank', 'paper_id', 'year', 'disruptiveness', 'disruption_class',
        'transe_unexpectedness', 'tgn_unexpectedness', 'composite_novelty']
cols = [c for c in cols if c in novel_out.columns]
print(novel_out[cols].head(10).to_string(index=False))

[15:02:46] Saving outputs...
[15:02:46]   ✓ novelty_feature_matrix_with_score.csv  (4,242 rows, 24 cols)
[15:02:46]   ✓ ..\..\outputs\final\transe_unexpected_scores.csv  (528 papers)
[15:02:46]   ✓ ..\..\outputs\final\tgn_unexpected_scores.csv    (789 papers)
[15:02:46]   ✓ ..\..\outputs\final\combined_disruptiveness.csv  (762 NOVEL papers)
[15:02:46]   ✓ transe_training_loss.csv

=== TOP 10 MOST DISRUPTIVE NOVEL PAPERS ===
 rank              paper_id  year  disruptiveness disruption_class  transe_unexpectedness  tgn_unexpectedness  composite_novelty
    1            NOVEL_SA_7  2021        1.000000       DISRUPTIVE               0.638298            0.629266           0.910588
    2           NOVEL_SA_14  2021        0.984084       DISRUPTIVE               0.674337            0.632430           0.884739
    3          NOVEL_DIA_49  2021        0.982190       DISRUPTIVE               1.000000            0.567933           0.826699
    4 NOVEL_DIA_W4386347795  2023        0.966867       

## Cell 9 — Validation & Sanity Checks

Verify correctness of outputs before passing to Step 5.

In [37]:
log('Running sanity checks...')

# 1. Score range
assert out['transe_unexpectedness'].between(0, 1).all(), 'TransE scores out of [0,1]'
assert out['tgn_unexpectedness'].between(0, 1).all(),    'TGN scores out of [0,1]'
assert out['disruptiveness'].between(0, 1).all(),        'Disruptiveness out of [0,1]'
log('  ✓ All scores in [0, 1]')

# 2. No NaN in key columns
for col in ['transe_unexpectedness', 'tgn_unexpectedness', 'kg_unexpectedness', 'disruptiveness']:
    assert out[col].notna().all(), f'NaN found in {col}'
log('  ✓ No NaN values in score columns')

# 3. Paper count preserved
assert len(out) == len(novelty_matrix), f'Row count changed: {len(out)} vs {len(novelty_matrix)}'
log(f'  ✓ Row count preserved: {len(out):,}')

# 4. NOVEL paper coverage
novel_in_matrix = out[out['paper_id'].str.startswith('NOVEL_')]
transe_coverage = (novel_in_matrix['transe_unexpectedness'] > 0).mean()
tgn_coverage    = (novel_in_matrix['tgn_unexpectedness'] > 0).mean()
log(f'  NOVEL paper TransE coverage: {transe_coverage*100:.1f}%')
log(f'  NOVEL paper TGN coverage   : {tgn_coverage*100:.1f}%')

# 5. Score distribution stats
print()
print('=== Score Distribution Summary ===')
print(out[['transe_unexpectedness', 'tgn_unexpectedness',
           'kg_unexpectedness', 'disruptiveness']].describe().round(4))

# 6. TransE loss convergence check
if len(loss_history) > 10:
    early = np.mean(loss_history[:5])
    late  = np.mean(loss_history[-5:])
    drop  = (early - late) / early * 100
    log(f'  TransE loss drop: {early:.2f} → {late:.2f}  ({drop:.1f}% reduction)')
    assert late < early, 'WARNING: TransE loss did not decrease — check hyperparameters'

log('All checks passed. Step 4 outputs are ready for Step 5 visualization.')

[15:02:46] Running sanity checks...
[15:02:46]   ✓ All scores in [0, 1]
[15:02:46]   ✓ No NaN values in score columns
[15:02:46]   ✓ Row count preserved: 4,242
[15:02:46]   NOVEL paper TransE coverage: 65.0%
[15:02:46]   NOVEL paper TGN coverage   : 92.1%

=== Score Distribution Summary ===
       transe_unexpectedness  tgn_unexpectedness  kg_unexpectedness  \
count              4242.0000           4242.0000          4242.0000   
mean                  0.0622              0.1161             0.0891   
std                   0.1790              0.2877             0.1980   
min                   0.0000              0.0000             0.0000   
25%                   0.0000              0.0000             0.0000   
50%                   0.0000              0.0000             0.0000   
75%                   0.0000              0.0000             0.0000   
max                   1.0000              1.0000             0.7840   

       disruptiveness  
count       4242.0000  
mean           0.631